# Partie 2 : VaR et Expected Shortfall avec trois méthodes

On estime la VaR et l'ES du portefeuille par trois méthodes (historique, gaussienne, KDE),
on les compare, puis on verifie leurs fiabilité sur des données simulées.

In [5]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t

from src.data import load_prices, compute_returns, portfolio_losses
from src.risk_measures import (var_hist, es_hist, var_gaussien, es_gaussien,
                               var_kde, es_kde, silverman_bandwidth)

tickers = ["TTE.PA", "BNP.PA", "MC.PA", "AIR.PA", "SAN.PA"]
prices = load_prices(tickers, "2005-01-01", "2025-12-31", "../data/prices.csv")
returns = compute_returns(prices)
losses = portfolio_losses(returns)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Les trois méthodes

- **Historique** : quantile empirique des pertes observées. Aucune hypothèse sur leur loi.
- **Gaussienne** : VaR = μ + z·σ, en supposant des pertes normales.
- **KDE** : la loi des pertes est estimée en lissant les observations par un noyau gaussien
  (largeur h fixée par la règle de Silverman), puis on en prend le quantile.

In [6]:
tableau = pd.DataFrame({
    "VaR 95 %": [var_hist(losses, 0.95), var_gaussien(losses, 0.95), var_kde(losses, 0.95)],
    "VaR 99 %": [var_hist(losses, 0.99), var_gaussien(losses, 0.99), var_kde(losses, 0.99)],
    "ES 97,5 %": [es_hist(losses, 0.975), es_gaussien(losses, 0.975), es_kde(losses, 0.975)],
    "ES 99 %": [es_hist(losses, 0.99), es_gaussien(losses, 0.99), es_kde(losses, 0.99)],
}, index=["Historique", "Gaussienne", "KDE"])

(tableau * 100).round(2)   # en %

,VaR 95 %,VaR 99 %,"ES 97,5 %",ES 99 %
Historique,2.18,4.20,4.39,5.76
Gaussienne,2.33,3.30,3.32,3.79
KDE,2.19,4.23,4.41,5.77


**Observation**
- **Historique et KDE donnent quasiment les mêmes valeurs** : la KDE ne fait que lisser les données réelles.
- **La gaussienne sous-estime le risque à 99 %** (VaR : 3,3 % contre 4,2 %, soit −21 % ;
  ES : 3,8 % contre 5,8 %, soit −34 %), car la loi normale ignore les queues épaisses.
- **À 95 %, elle le surestime légèrement** : l'erreur de la loi normale change de sens selon le niveau.
  Plus on va loin dans la queue, plus elle sous-estime.



## 2. Vérification sur données simulées

Sur les vraies données, la VaR exacte est inconnue. On tire donc 1 000 échantillons de 500 pertes
d'une loi de Student à 4 degrés de liberté (queues épaisses, VaR 99 % exacte connue),
et on regarde quelle méthode la retrouve.

In [7]:
n=1000
liste_var=[]
liste_gaus=[]
liste_kde=[]

rng=np.random.default_rng(42)

vraie_var = t.ppf(0.99, df=4)

for i in range(n):
    fausses_pertes=pd.Series(rng.standard_t(df=4,size=500))
    liste_var.append(var_hist(fausses_pertes, 0.99))            
    liste_gaus.append(var_gaussien(fausses_pertes, 0.99))       
    liste_kde.append(var_kde(fausses_pertes, 0.99)) 

print("Vraie VaR :", vraie_var)
print("Moyenne hist  :", np.mean(liste_var),  "  dispersion :", np.std(liste_var))
print("Moyenne gauss :", np.mean(liste_gaus), "  dispersion :", np.std(liste_gaus))
print("Moyenne KDE   :", np.mean(liste_kde),   "  dispersion :", np.std(liste_kde))

Vraie VaR : 3.746947387979196
Moyenne hist  : 3.640095979266116   dispersion : 0.47811488143648034
Moyenne gauss : 3.269315521409218   dispersion : 0.26703767827500163
Moyenne KDE   : 3.8003992956700046   dispersion : 0.48446260243969447


**Résultats** (vraie VaR 99 % de la loi de Student à 4 degrés de liberté : **3,75**)

| Méthode | Moyenne des 1 000 VaR estimées | Biais | Dispersion (écart-type) |
|---|---|---|---|
| Historique | 3,64 | −2,9 % | 0,48 |
| KDE | 3,80 | +1,4 % | 0,48 |
| Gaussienne | 3,27 | **−12,7 %** | 0,27 |

**Interprétation**

- **La VaR gaussienne est biaisée vers le bas** : en moyenne, elle sous-estime la vraie VaR de près de 13 %,
  et toujours dans le même sens. Ce n'est pas de la malchance : la loi normale n'a pas de queues épaisses,
  elle ne peut donc pas capter les pertes extrêmes. Sa faible dispersion est trompeuse : elle est
  **régulièrement fausse**.
- **Les VaR historique et KDE sont quasi sans biais** (−2,9 % et +1,4 %) : elles respectent la forme réelle
  des données et retrouvent en moyenne la bonne valeur.
- **Mais elles sont plus dispersées** (0,48, soit environ 13 % de la vraie valeur) : avec 500 observations,
  la VaR à 99 % ne repose que sur environ 5 pertes extrêmes, et l'estimation varie beaucoup d'un échantillon
  à l'autre.

**Conclusion** : Un biais vers le bas est le défaut le plus grave, car il sous-estime systématiquement les pertes. Les méthodes historique et KDE sont préférables, mais leur
imprécision dans les queues motive l'utilisation de la théorie des valeurs extrêmes (partie 4).